# GRACE H1 — Bắt đầu full pipeline tại đây

Đây là notebook điều phối cho **local → Kaggle → Colab CPU → local**. Mở notebook này trước và chạy từ trên xuống. Khi cần chuyển môi trường, cell dừng bằng RuntimeError có hướng dẫn; tải output về bundles/ rồi Run All lại. Không cần đoán cell nào được chạy tiếp.

| Bước | Chạy ở đâu | File cần mang đi / mang về |
|---|---|---|
| 1. Train projection CE + SupCon/hard | Kaggle GPU | training-inputs.zip → trained-heads.zip |
| 2. Whitening, retrieval, Joern, rerank, frozen prompts | Colab CPU, accelerator None | cpu-inputs.zip + joern-runtime.zip + trained-heads.zip → cpu-prepared.zip |
| 3. Preflight, Vertex LLM, metrics/paired CI | Local | Import heads + prepared; dùng response cache/ADC đã có |

Nhánh reveal mặc định chạy toàn bộ **2.274 mẫu test ReVeal**, hai arm baseline/H1, seed 42. Kết quả Devign được giữ trên nhánh devign; artifact local nằm trong archives/devign/2026-10-16/. Ba notebook trên nhánh reveal đã xóa output Devign để bắt đầu lượt mới. Embedding, graph và response cache dùng chung được giữ để tái sử dụng.

Upload training-inputs.zip mới cho ReVeal lên Kaggle. Colab dùng **MyDrive/GRACE-H1-transfer-reveal** và runtime mới để không nhập lại checkpoint Devign.

- Kaggle: [kaggle_train_projection.ipynb](./kaggle_train_projection.ipynb).
- Colab: [colab_cpu_prepare.ipynb](./colab_cpu_prepare.ipynb).
- Hướng dẫn chuyển file: [CLOUD_HANDOFF.md](../docs/CLOUD_HANDOFF.md).

In [23]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util
candidates=([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h1').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H1_ROOT to the local project folder')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=True
TRAIN_ON_KAGGLE=True
PREPARE_ON_COLAB=False
RUN_INFERENCE=True  # Calls Vertex for prompts missing from cache, only after all handoffs pass.
HEADS_BUNDLE=ROOT/'bundles'/'grace-h1-trained-heads.zip'
PREPARED_BUNDLE=ROOT/'bundles'/'grace-h1-cpu-prepared.zip'
config=json.loads((ROOT/'configs'/'experiment.json').read_text(encoding='utf-8'))
if config['datasets'] != ['reveal']: raise ValueError('Nhánh reveal yêu cầu datasets=[reveal]; kiểm tra configs/experiment.json')
print('Project:',ROOT)
print(json.dumps(config,indent=2))

Project: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection
{
  "datasets": [
    "reveal"
  ],
  "splits": [
    "test"
  ],
  "objectives": [
    "ce",
    "supcon_hard"
  ],
  "llm_arms": [
    "baseline_whitening",
    "h1_supcon_hard"
  ],
  "training": {
    "dimension": 128,
    "epochs": 30,
    "batch_size": 256,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "supcon_weight": 0.1,
    "hard_negative_weight": 0.1,
    "temperature": 0.1,
    "margin": 0.2,
    "class_weighted_ce": true,
    "patience": 6,
    "seed": 42,
    "device": "auto"
  },
  "pipeline": {
    "semantic_top_k": 5,
    "lexical_weight": 0.7,
    "whitening_dimension": 256,
    "graph_workers": 3,
    "joern_home": null
  },
  "inference_workers": 4,
  "bootstrap": 1000
}


In [24]:
if INSTALL_DEPENDENCIES:
    if TRAIN_ON_KAGGLE:
        # Local handoff/inference does not need PyTorch or CUDA.
        subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt'),'google-genai>=1.0,<3'])
    else:
        if importlib.util.find_spec('torch') is None:
            subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.6,<3','--index-url','https://download.pytorch.org/whl/cpu'])
        subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])
from grace_h1.io import read_json,load_local_environment
from grace_h1.portable import create_bundle,import_bundle
from grace_h1.offload import require_trained_heads,require_prepared_runs
from grace_h1.experiment import train_all,prepare_all,preflight_all,infer_all,evaluate_all
import pandas as pd
load_local_environment(ROOT)

## Bước 1 — Head trên Kaggle

**Lần đầu:** cell dưới tạo training-inputs bundle rồi dừng. Đính kèm bundle vào Kaggle Dataset, upload kaggle_train_projection.ipynb, bật GPU/Internet, Run All. Tải grace-h1-trained-heads.zip về bundles/ local rồi Run All notebook này lại.

**Nếu đã có head khớp:** import/reuse, không train trên local. Head/config/source/data sai fingerprint sẽ dừng để xuất lại đúng bộ. TRAIN_ON_KAGGLE=False chỉ dùng khi chủ động muốn train local.

In [25]:
if HEADS_BUNDLE.exists(): print(import_bundle(ROOT,HEADS_BUNDLE,'trained_heads'))
if TRAIN_ON_KAGGLE:
    try:
        selection=require_trained_heads(ROOT,config)
    except (FileNotFoundError,KeyError,ValueError) as exc:
        info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-training-inputs.zip','training_inputs')
        print(info)
        raise RuntimeError('DỪNG BƯỚC 1: mở notebooks/kaggle_train_projection.ipynb trên Kaggle; đính kèm training-inputs.zip; tải trained-heads.zip về bundles/ rồi Run All lại. Chi tiết head: '+str(exc)) from exc
else:
    selection=train_all(ROOT,config)
    print(create_bundle(ROOT,HEADS_BUNDLE,'trained_heads'))
display(pd.DataFrame([{'dataset':d,'objective':o,'parameters':h['parameters'],'best_epoch':h['best']['epoch'],
                      'validation_mcc':h['best']['metrics']['mcc'],'trained_device':h['device']}
                     for d,heads in selection['heads'].items() for o,h in heads.items()]))

{'kind': 'trained_heads', 'copied': 0, 'reused_identical': 11, 'archive_sha256': '5800350161584e3b1fb3737669538fb650dad0d05d8a975623cf96e43aa30162'}


,dataset,objective,parameters,best_epoch,validation_mcc,trained_device
0,reveal,ce,98690,11,0.302242,cuda
1,reveal,supcon_hard,98690,11,0.300041,cuda


## Bước 2 — CPU/RAM: dùng kết quả preparation local WSL

Preparation cho `reveal` đã chạy trong `notebooks/wsl_home_prepare.ipynb` trên WSL `/home` và bundle kết quả đã được nhập vào `runs/` của project này. Cell kế tiếp kiểm tra fingerprint và dùng lại hai frozen run; không cần chạy lại Colab.

Nếu chủ động cần chạy preparation trên Colab, đổi `PREPARE_ON_COLAB=True` ở cell cấu hình phía trên. Khi không tìm thấy frozen run hợp lệ, notebook sẽ tạo `cpu-inputs.zip` và hướng dẫn chuyển sang `notebooks/colab_cpu_prepare.ipynb`.


In [26]:
if PREPARED_BUNDLE.exists():
    print(import_bundle(ROOT,PREPARED_BUNDLE,'cpu_prepared'))
try:
    run_rows=require_prepared_runs(ROOT,config)
    print('Dùng lại frozen CPU preparation đã có; không cần chạy Colab/prepare lại.')
except (FileNotFoundError,KeyError,ValueError) as exc:
    if PREPARE_ON_COLAB:
        info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-inputs.zip','cpu_inputs')
        print(info)
        runtime=ROOT/'bundles'/'grace-h1-joern-runtime.zip'
        print('Joern runtime:',runtime,'exists:',runtime.exists())
        raise RuntimeError('DỪNG BƯỚC 2: chưa có CPU preparation hợp lệ. Nếu muốn offload, mở notebooks/colab_cpu_prepare.ipynb trên Colab CPU; upload cpu-inputs.zip, joern-runtime.zip, trained-heads.zip; tải cpu-prepared.zip về bundles/ rồi chạy lại. Chi tiết: '+str(exc)) from exc
    run_rows=prepare_all(ROOT,config)
display(pd.DataFrame(run_rows))


Dùng lại frozen CPU preparation đã có; không cần chạy Colab/prepare lại.


,arm,dataset,path,split
0,baseline_whitening,reveal,runs/reveal/test/baseline_whitening/0779c879b6...,test
1,h1_supcon_hard,reveal,runs/reveal/test/h1_supcon_hard/d318d1344264b913,test


## Bước 3 — Local: preflight trước khi gọi Vertex

Đã có head và frozen prompts từ cloud. ADC/Vertex chỉ dùng tại local. Exact prompt/model/decoding khớp sẽ reuse response cũ; tổng new requests từng arm là cận trên vì prompt trùng giữa các arm tiếp tục được reuse khi chạy tuần tự. RUN_INFERENCE=False chỉ đọc cache.

RUN_INFERENCE=True ở cell cấu hình: khi chạy tiếp cell inference bên dưới sẽ gọi API cho phần thiếu.

In [27]:
plans=preflight_all(ROOT,config)
display(pd.DataFrame([{k:p[k] for k in ['dataset','split','arm','queries','cached_queries','unique_prompts','new_unique_requests','worst_case_attempts']} for p in plans]))

,dataset,split,arm,queries,cached_queries,unique_prompts,new_unique_requests,worst_case_attempts
0,reveal,test,baseline_whitening,2274,0,2270,2270,11350
1,reveal,test,h1_supcon_hard,2274,0,2270,2270,11350


In [28]:
results=infer_all(ROOT,config,allow_api=RUN_INFERENCE)
display(pd.DataFrame([{k:r.get(k) for k in ['dataset','split','arm','status','cached_queries','new_unique_requests','api_attempts_this_run','request_error_queries','wall_seconds']} for r in results]))

LLM unique requests 25/2270
LLM unique requests 50/2270
LLM unique requests 75/2270
LLM unique requests 100/2270
LLM unique requests 125/2270
LLM unique requests 150/2270
LLM unique requests 175/2270
LLM unique requests 200/2270
LLM unique requests 225/2270
LLM unique requests 250/2270
LLM unique requests 275/2270
LLM unique requests 300/2270
LLM unique requests 325/2270
LLM unique requests 350/2270
LLM unique requests 375/2270
LLM unique requests 400/2270
LLM unique requests 425/2270
LLM unique requests 450/2270
LLM unique requests 475/2270
LLM unique requests 500/2270
LLM unique requests 525/2270
LLM unique requests 550/2270
LLM unique requests 575/2270
LLM unique requests 600/2270
LLM unique requests 625/2270
LLM unique requests 650/2270
LLM unique requests 675/2270
LLM unique requests 700/2270
LLM unique requests 725/2270
LLM unique requests 750/2270
LLM unique requests 775/2270
LLM unique requests 800/2270
LLM unique requests 825/2270
LLM unique requests 850/2270
LLM unique reques

,dataset,split,arm,status,cached_queries,new_unique_requests,api_attempts_this_run,request_error_queries,wall_seconds
0,reveal,test,baseline_whitening,complete,0,2270,2270,0,504.822489
1,reveal,test,h1_supcon_hard,complete,1934,340,340,0,97.495447


## Bước 4 — Metrics và paired comparison

Phần tổng hợp nhỏ chạy local sau LLM. F1/MCC trên valid predictions, luôn báo invalid rate và recall trên mọi positive. Paired bootstrap dùng cùng query valid ở hai arm. Notebook chưa có kết quả ReVeal; chỉ kết luận sau khi hoàn tất inference và evaluation.

In [29]:
report=evaluate_all(ROOT,config)
rows=[]
for r in report['scores']:
    m=r['metrics'];i=r['inference']
    rows.append({'dataset':r['dataset'],'split':r['split'],'arm':r['arm'],'n':m['n'],'f1_valid':m['f1'],
                 'mcc_valid':m['mcc'],'recall_all_positives':m['recall_all_positives'],'invalid_rate':m['invalid_rate'],
                 'new_requests':i['new_unique_requests'],'api_attempts':i['api_attempts_this_run'],
                 'new_usage':i['provider_usage_new_requests']})
display(pd.DataFrame(rows))
print('Status:',report['status'])
for comparison in report['comparisons']: print(json.dumps(comparison,indent=2))
print('Saved:',ROOT/'runs'/'experiment_summary.json')

,dataset,split,arm,n,f1_valid,mcc_valid,recall_all_positives,invalid_rate,new_requests,api_attempts,new_usage
0,reveal,test,baseline_whitening,2274,0.171098,0.031775,0.321739,0.0,2270,2270,"{'candidates_token_count': 4540, 'prompt_token..."
1,reveal,test,h1_supcon_hard,2274,0.166090,0.024591,0.313043,0.0,340,340,"{'candidates_token_count': 680, 'prompt_token_..."


Status: complete
{
  "common_valid_n": 2274,
  "excluded_invalid_pair_n": 0,
  "delta_right_minus_left": {
    "f1": -0.005008300498029833,
    "mcc": -0.007183179349265031,
    "recall": -0.008695652173912993,
    "precision": -0.003505605755324534
  },
  "ci95_query_bootstrap": {
    "f1": [
      -0.01593440452146734,
      0.0052092765159745796
    ],
    "mcc": [
      -0.022561295271047166,
      0.007203774810143289
    ],
    "recall": [
      -0.029858787956767885,
      0.012827510917030572
    ],
    "precision": [
      -0.011039225939262017,
      0.003404373755291241
    ]
  },
  "bootstrap_repetitions": 1000,
  "note": "query bootstrap; repeated code/project dependence is not cluster-adjusted",
  "left": "runs/reveal/test/baseline_whitening/0779c879b6d11431",
  "right": "runs/reveal/test/h1_supcon_hard/d318d1344264b913"
}
Saved: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/runs/experiment_summary.json
